# Imports

In [1]:
import json
import requests

import pandas as pd
import xml.etree.ElementTree as ET

from pathlib import Path

# Params

In [2]:
PROJECT_ROOT = Path.cwd().parent

OUTPUT_DIR = PROJECT_ROOT / "data" / "raw"

QUERY_PATH = OUTPUT_DIR / "arxiv_query.json"

In [3]:
with open(
    QUERY_PATH,
    encoding="utf-8",
) as file:
    query = json.load(file)

QUERY = query["query"]

# QUERY = "multi agent systems"

MAX_RESULTS = 10

In [4]:
BASE_URL = "http://export.arxiv.org/api/query"

params = {
    "search_query": f"all:{QUERY}",
    "start": 0,
    "max_results": MAX_RESULTS,
    "sortBy": "submittedDate",
    "sortOrder": "descending",
}

In [5]:
params

{'search_query': 'all:(all:"Covid-19" OR all:"SARS-CoV-2") AND (all:"vaccine validation" OR all:"vaccine efficacy")',
 'start': 0,
 'max_results': 10,
 'sortBy': 'submittedDate',
 'sortOrder': 'descending'}

# Call API

In [6]:
response = requests.get(
    BASE_URL,
    params=params,
    timeout=30,
)

response.raise_for_status()

print(response.status_code)

200


In [7]:
print(response.text[:1500])

<?xml version='1.0' encoding='UTF-8'?>
<feed xmlns:opensearch="http://a9.com/-/spec/opensearch/1.1/" xmlns:arxiv="http://arxiv.org/schemas/atom" xmlns="http://www.w3.org/2005/Atom">
  <id>https://arxiv.org/api/fK1NoB76wp1jKal9WSvHV86IncU</id>
  <title>arXiv Query: search_query=(all:"Covid-19" OR all:"SARS-CoV-2") AND (all:"vaccine validation" OR all:"vaccine efficacy")&amp;id_list=&amp;start=0&amp;max_results=10</title>
  <updated>2026-08-03T11:26:07Z</updated>
  <link href="https://arxiv.org/api/query?search_query=(all:%22Covid-19%22+OR+all:%22SARS-CoV-2%22)+AND+(all:%22vaccine+validation%22+OR+all:%22vaccine+efficacy%22)&amp;start=0&amp;max_results=10&amp;id_list=" type="application/atom+xml"/>
  <opensearch:itemsPerPage>10</opensearch:itemsPerPage>
  <opensearch:totalResults>47</opensearch:totalResults>
  <opensearch:startIndex>0</opensearch:startIndex>
  <entry>
    <id>http://arxiv.org/abs/2604.13265v1</id>
    <title>Efficient estimation of cumulative incidence curves via data fu

In [8]:
root = ET.fromstring(response.text)

In [9]:
namespace = {
    "atom": "http://www.w3.org/2005/Atom"
}

In [10]:
entries = root.findall(
    "atom:entry",
    namespace,
)

len(entries)

10

# Build a table from response

In [11]:
papers = []

for entry in entries:

    title = entry.find(
        "atom:title",
        namespace,
    ).text.strip()

    summary = entry.find(
        "atom:summary",
        namespace,
    ).text.strip()

    published = entry.find(
        "atom:published",
        namespace,
    ).text

    url = entry.find(
        "atom:id",
        namespace,
    ).text

    authors = [
        author.find(
            "atom:name",
            namespace,
        ).text
        for author in entry.findall(
            "atom:author",
            namespace,
        )
    ]

    papers.append(
        {
            "title": title,
            "authors": ", ".join(authors),
            "published": published,
            "summary": summary,
            "url": url,
        }
    )

In [12]:
df = pd.DataFrame(papers)

In [13]:
df

,title,authors,published,summary,url
0,Efficient estimation of cumulative incidence c...,"Pan Zhao, Peter B. Gilbert, Oliver Dukes, Bo Z...",2026-04-14T19:53:16Z,Refined vaccine regimens containing variant-ma...,http://arxiv.org/abs/2604.13265v1
1,A simple and powerful test of vaccine waning,"Gellért Perényi, Matias Janvin, Mats J. Stensrud",2025-11-26T19:06:15Z,Determining whether vaccine efficacy wanes is ...,http://arxiv.org/abs/2511.21836v2
2,"Debiasing hazard-based, time-varying vaccine e...","Ethan Ashby, Dean Follmann, Holly Janes, Peter...",2025-11-19T04:15:15Z,Understanding how vaccine effectiveness (VE) c...,http://arxiv.org/abs/2511.15099v1
3,Nonparametric bounds for vaccine effects in ra...,"Rachel Axelrod, Uri Obolski, Daniel Nevo",2025-10-29T09:00:30Z,Vaccine randomized trials are typically design...,http://arxiv.org/abs/2510.25296v2
4,Sequentially Doubly Robust Estimation of Condi...,"Hongxiang Qiu, Marco Carone, Alex Luedtke, Pet...",2025-10-12T00:00:55Z,It is often of interest to study the associati...,http://arxiv.org/abs/2510.10372v4
5,Sensitivity analysis of an epidemic model with...,Ma. Cristina R. Bargo,2025-09-04T13:07:43Z,The COVID-19 pandemic forced the rapid develop...,http://arxiv.org/abs/2509.04188v1
6,Evaluation of Surrogate Endpoints Based on Met...,"Florian Stijven, Peter B. Gilbert",2025-09-01T19:43:30Z,The meta-analytic (MA) framework is the gold s...,http://arxiv.org/abs/2509.01737v2
7,Test-Negative Designs with Multiple Testing So...,"Mengxin Yu, Nicholas P. Jewell",2025-04-28T13:24:52Z,"Test-negative designs (TNDs), a form of case-c...",http://arxiv.org/abs/2504.19778v1
8,Target trial emulation without matching: a mor...,"Emily Wu, Elizabeth Rogawski McQuade, Mats Ste...",2025-04-23T21:32:21Z,Real-world vaccine effectiveness has increasin...,http://arxiv.org/abs/2504.17104v1
9,Optimal COVID-19 vaccine prioritization by age...,"Iker Atienza-Diez, Gabriel Rodriguez-Maroto, S...",2025-02-26T16:51:19Z,The limited availability of COVID-19 vaccines ...,http://arxiv.org/abs/2502.19292v1


# Save results

In [14]:
df.to_csv(
    OUTPUT_DIR / "arxiv_search.csv",
    index=False,
)